In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Read the dataset Q1_data.csv using read_csv()
import pandas as pd
import os

deliv_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(deliv_path)

In [ ]:
# Task 2: Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Display dataset information using info()
df.info()

In [ ]:
# Task 4: Show statistical description using describe()
df.describe()

In [ ]:
# Task 5: Plot the target distribution (delivery_time)
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution (Target)')
plt.xlabel('Time')
plt.ylabel('Orders')
plt.show()

In [ ]:
# Task 1: Drop the 'Order_ID' column from the data
df = df.drop(columns=['Order_ID'])
df.head()

In [ ]:
# Task 2: Handle missing values appropriately
# (Hint: I guess you want to have a closer look at the columns with missing values :) )

# Analyze missing values
def analyze_missing(df):
    missing_percentage = (df.isnull().sum() / len(df)) * 100

    missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
    })

    missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

    return missing_data

print("Missing Data Analysis:")
missing_data = analyze_missing(df)
missing_data.head(10)

In [ ]:
# Drop all missing values as they're not much
df = df.dropna(subset=['Delivery_Time', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs'])

checking_missing_data = analyze_missing(df)
checking_missing_data.head(10)

In [ ]:
# Task 3: Check and remove duplicates if any exist
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)
le = LabelEncoder()

df['Weather'] = le.fit_transform(df['Weather'])
df['Traffic_Level'] = le.fit_transform(df['Traffic_Level'])
df['Time_of_Day'] = le.fit_transform(df['Time_of_Day'])
df['Vehicle_Type'] = le.fit_transform(df['Vehicle_Type'])

df.head()

In [ ]:
# Task 5: Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

features = df.columns.drop('Delivery_Time')

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)
# It's Regression Task (:

In [ ]:
# Task 1: Split the dataset into features (X) and target (y)
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

In [ ]:
# # Task 2: Use the correct split: KFold OR StratifiedKFold
# from sklearn.model_selection import train_test_split, KFold
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_absolute_error

# model = RandomForestRegressor()

# # I'll use KFold as it's regression task
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# mae_scores = []

# for train_idx, test_idx in kf.split(X):
#     X_train, X_test = X[train_idx], X[test_idx]
#     y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

#     # Task 3: Train a RandomForest model
#     model.fit(X_train, y_train)

#     # Predict
#     y_pred = model.predict(X_test)

#     # Task 4: Evaluate using MAE (Mean Absolute Error) ONLY
#     mae_scores.append(mean_absolute_error(y_test, y_pred))

In [ ]:
# from sklearn.model_selection import train_test_split, KFold
# from sklearn.ensemble import RandomForestRegressor
# import numpy as np

# X_train, X_test, y_train, y_test = train_test_split(X, y)

# kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# model = RandomForestRegressor()

# mae_scores = []

# for train_idx, val_idx in kfold.split(X_train):
#     X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
#     y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

#     # Train and predict
#     model.fit(X_fold_train, y_fold_train)
#     y_fold_pred = model.predict(X_fold_val)

#     # Calculate metrics
#     mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

# mae_scores = np.array(mae_scores)

# print(f"5-Fold CV Results:")
# print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#  Task 3: Train a RandomForest model
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
# Task 4: Evaluate using MAE (Mean Absolute Error) ONLY
from sklearn.metrics import mean_absolute_error

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE:  {mae:,.2f}")

In [ ]:
# Task 5: Print the averaged score across all folds

In [ ]:
# Task 1: Plot feature importance from your trained model
feature_importance = pd.DataFrame({
    'feature': df.columns.drop('Delivery_Time'),
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Plot predicted delivery time histogram
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('Predicted Delivery Time Distribution')
plt.xlabel('Time')
plt.ylabel('Orders')
plt.show()

In [ ]:
# Task Bonus: Write your code here: